# Get onboard refunds quantity and amount for a specific date interval
per driver,
per trip and date
per agency

## Configuration

In [ ]:
# =========================
# FILTERS FOR DATA EXTRACTION
# =========================
START_DATE = "20260201"
END_DATE   = "20260228"

# =========================
# VARS FOR PARALLEL PROCESSING
# =========================
MAX_WORKERS = 10
CHUNK_SIZE_DAYS = 3

# =========================
# GLOBAL VARS TO GET FROM CONFIG FILE
# =========================

required_vars = [
    "MONGO_URI",
    "DB_NAME",
    "COLLECTION_NAME",  #rideones
    "TIMEZONE",
    "OUTPUT_FILES_FOLDER",
]



In [ ]:
# from config py file

import importlib
import config

# import, cleaning the cache to get latest changes
importlib.reload(config)


print("Config file in:", config.__file__)


missing = []
locals_dict = locals()

for name in required_vars:
    if hasattr(config, name):
        locals_dict[name] = getattr(config, name)
    else:
        missing.append(name)

if missing:
    raise RuntimeError(f"Missing config variables: {missing}")

# Create directory for output files if it doesn't exist
import os
from pathlib import Path
OUTPUT_FILES_FOLDER.mkdir(parents=True, exist_ok=True)

[name for name in required_vars if hasattr(config, name) and print(name, getattr(config, name))]

## Get rides

In [ ]:
# =========================
# Imports
# =========================
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from pymongo import MongoClient
import pandas as pd

# =========================
# MongoDB connection (GLOBAL)
# =========================
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# =========================
# Generate date chunks
# =========================
def generate_chunks(start, end, size):
    """
    Generates inclusive date chunks with no overlap and no gaps.
    START_DATE and END_DATE are both included.
    """
    start_dt = datetime.strptime(start, "%Y%m%d")
    end_dt = datetime.strptime(end, "%Y%m%d")

    chunks = []
    cur = start_dt
    while cur <= end_dt:
        nxt = min(cur + timedelta(days=size - 1), end_dt)
        chunks.append({
            "start": cur.strftime("%Y%m%d"),
            "end": nxt.strftime("%Y%m%d")
        })
        cur = nxt + timedelta(days=1)
    return chunks

date_chunks = generate_chunks(START_DATE, END_DATE, CHUNK_SIZE_DAYS)
print(f"Chunks: {len(date_chunks)}")

# =========================
# Fetch per chunk (FIELDS ONLY)
# =========================
def fetch_chunk(chunk):
    pipeline = [
        {
            "$match": {
                "operational_date": {
                    "$gte": chunk["start"],
                    "$lte": chunk["end"]  # inclusive, matches chunk logic
                },
                "agency_id": {"$in": ["41", "42", "43", "44"]}
            }
        },
        {
            "$project": {
                "_id": 1,
                "agency_id": 1,
                "line_id": 1,
                #"route_id": 1,
                "trip_id": 1,
                #"three_vehicle_events_grade": "$analysis.SIMPLE_THREE_VEHICLE_EVENTS.grade",
                #"expected_start_time_grade": "$analysis.EXPECTED_START_TIME.grade",
                #"end_time_observed": 1,
                #"extension_observed": 1,
                #"extension_scheduled": 1,
                "operational_date": 1,
                #"start_time_observed": 1,
                #"start_time_scheduled": 1,
                #"end_time_scheduled": 1,
                "passengers_observed": 1,
                "passengers_observed_on_board_sales_qty": 1,
                "driver_ids": "$driver_ids",
                "apex_on_board_refunds_qty": 1,
                "apex_on_board_refunds_amount": 1,
            }
        }
    ]

    return list(collection.aggregate(pipeline, allowDiskUse=True))

# =========================
# Parallel execution
# =========================
rows = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(fetch_chunk, c) for c in date_chunks]

    for i, f in enumerate(as_completed(futures), 1):
        chunk_rows = f.result()
        rows.extend(chunk_rows)
        print(f"Processed {i}/{len(date_chunks)} chunks | rows so far: {len(rows)}")

# =========================
# Final DataFrame
# =========================
rides_df2 = pd.DataFrame(rows)


In [ ]:
rides_df2.apex_on_board_refunds_qty.sort_values(ascending=False).unique()

In [ ]:
rides_df2.head(3)

## Agregate by trip_date_driver

In [ ]:
import pandas as pd

# 1. Explode driver_ids into separate rows
df = rides_df2.explode("driver_ids")

# Optional: rename for clarity
df = df.rename(columns={"driver_ids": "driver_id"})

# 2. Aggregate
aggregated_trip_date_driver_df = (
    df.groupby(
        [
            "driver_id",
            "agency_id",
            "line_id",
            "trip_id",
            "operational_date",
        ],
        as_index=False
    )
    .agg(
        apex_on_board_refunds_qty=("apex_on_board_refunds_qty", "sum"),
        apex_on_board_refunds_amount=("apex_on_board_refunds_amount", "sum"),
        passengers_observed=("passengers_observed", "sum"),
        passengers_observed_on_board_sales_qty=("passengers_observed_on_board_sales_qty", "sum"),
    )
.sort_values(by="apex_on_board_refunds_qty", ascending=False)
)

aggregated_trip_date_driver_df["apex_on_board_refunds_amount_eur"] = aggregated_trip_date_driver_df["apex_on_board_refunds_amount"] /100

In [ ]:
aggregated_trip_date_driver_df["driver_unique_id"] = aggregated_trip_date_driver_df["agency_id"] + "_" + aggregated_trip_date_driver_df["driver_id"]
aggregated_trip_date_driver_df["on_board_sales_and_refunds"] = aggregated_trip_date_driver_df["passengers_observed_on_board_sales_qty"] + aggregated_trip_date_driver_df["apex_on_board_refunds_qty"]


In [ ]:
aggregated_trip_date_driver_df.head()

In [ ]:
#export csv
output_file = OUTPUT_FILES_FOLDER / f"aggregated_refunds_by_driver_trip_date_{START_DATE}_{END_DATE}.csv"
aggregated_trip_date_driver_df.to_csv(output_file, index=False)

## Agregate refunds by Trip and Date

In [ ]:
# 2. Aggregate
aggregated_trip_date = (
    aggregated_trip_date_driver_df.groupby(
        [
            "agency_id",
            "line_id",
            "trip_id",
            "operational_date",
        ],
        as_index=False
    )
    .agg(
        apex_on_board_refunds_qty=("apex_on_board_refunds_qty", "sum"),
        apex_on_board_refunds_amount_eur=("apex_on_board_refunds_amount_eur", "sum"),
        passengers_observed=("passengers_observed", "sum"),
        passengers_observed_on_board_sales_qty=("passengers_observed_on_board_sales_qty", "sum"),
        on_board_sales_and_refunds=("on_board_sales_and_refunds", "sum"),

    )

)

import numpy as np
aggregated_trip_date["pct_refunds_from_on_board_sales_and_refunds"] = (aggregated_trip_date["apex_on_board_refunds_qty"] / aggregated_trip_date["on_board_sales_and_refunds"].replace(0, np.nan)).round(4)

aggregated_trip_date.sort_values(by="pct_refunds_from_on_board_sales_and_refunds", ascending=False)


In [ ]:
#export csv
output_file = OUTPUT_FILES_FOLDER / f"aggregated_refunds_trip_date_{START_DATE}_{END_DATE}.csv"
aggregated_trip_date.to_csv(output_file, index=False)

## Agregate by driver

In [ ]:
# Aggregate

import numpy as np

aggregated_driver_df = (
    aggregated_trip_date_driver_df.groupby(
        [
            "agency_id",
            "driver_id",
            "driver_unique_id",

        ],
        as_index=False
    )
    .agg(
        apex_on_board_refunds_qty=("apex_on_board_refunds_qty", "sum"),
        apex_on_board_refunds_amount_eur=("apex_on_board_refunds_amount_eur", "sum"),
        passengers_observed=("passengers_observed", "sum"),
        passengers_observed_on_board_sales_qty=("passengers_observed_on_board_sales_qty", "sum"),
        on_board_sales_and_refunds=("on_board_sales_and_refunds", "sum"),
    )
.sort_values(by="apex_on_board_refunds_qty", ascending=False)
)


aggregated_driver_df["pct_refunds_from_on_board_sales_and_refunds"] = (aggregated_driver_df["apex_on_board_refunds_qty"] / aggregated_driver_df["on_board_sales_and_refunds"].replace(0, np.nan)).round(4)



aggregated_driver_df.head(3)

In [ ]:
aggregated_driver_with_refunds_df = aggregated_driver_df[aggregated_driver_df["apex_on_board_refunds_qty"] > 0]
aggregated_driver_with_refunds_df.head(3)

In [ ]:
#export csv
output_file = OUTPUT_FILES_FOLDER / f"aggregated_driver_with_refunds_{START_DATE}_{END_DATE}.csv"
aggregated_driver_with_refunds_df.to_csv(output_file, index=False)

## Distributions visuals

In [ ]:
# Top 10 drivers with more refunds quantity
aggregated_driver_with_refunds_df = aggregated_driver_with_refunds_df.sort_values(
    by="pct_refunds_from_on_board_sales_and_refunds",
    ascending=False
)

aggregated_driver_with_refunds_df.head(20)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.boxplot(aggregated_driver_with_refunds_df["apex_on_board_refunds_qty"])
plt.ylabel("Apex On Board Refunds Quantity")
plt.title("Distribution of Refund Quantity per Driver")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.boxplot(aggregated_driver_with_refunds_df["pct_refunds_from_on_board_sales_and_refunds"])
plt.ylabel("Percentage On Board Refunds Qty \n from On Board Sales and Refunds")
plt.title("Distribution of Percentage of Refunds per Driver")
plt.show()

## Calculate Fiscalization Thresholds

In [ ]:
import pandas as pd

# Assume your aggregated_driver_df exists
qty_counts = aggregated_driver_with_refunds_df["apex_on_board_refunds_qty"]

# Basic quartiles
min_val = qty_counts.min()
q1 = qty_counts.quantile(0.25)
median = qty_counts.median()
q3 = qty_counts.quantile(0.75)

# Interquartile range (IQR)
iqr = q3 - q1

# Tukey-style fences
fence_1_5 = q3 + 1.5 * iqr
fence_3 = q3 + 3 * iqr

# Display results
summary_qty = pd.DataFrame({
    "Statistic": ["min", "Q1", "median", "Q3", "Q3 + 1.5*IQR", "Q3 + 3*IQR"],
    "Value_on_board_refunds_qty": [min_val, q1, median, q3, fence_1_5, fence_3]
})

summary_qty


In [ ]:
#export csv
output_file_qty = OUTPUT_FILES_FOLDER / f"summary_refund_qty_thresholds_per_driver_for_fiscalization_{START_DATE}_{END_DATE}.csv"
summary_qty.to_csv(output_file_qty, index=False)

In [ ]:
import pandas as pd


amount = aggregated_driver_with_refunds_df["pct_refunds_from_on_board_sales_and_refunds"]

# Basic quartiles
min_val = amount.min()
q1 = amount.quantile(0.25)
median = amount.median()
q3 = amount.quantile(0.75)

# Interquartile range (IQR)
iqr = q3 - q1

# Tukey-style fences
fence_1_5 = q3 + 1.5 * iqr
fence_3 = q3 + 3 * iqr

# Display results
summary_amount = pd.DataFrame({
    "Statistic": ["min", "Q1", "median", "Q3", "Q3 + 1.5*IQR", "Q3 + 3*IQR"],
    "Value_pct_refunds_from_on_board_sales_and_refunds": [min_val, q1, median, q3, fence_1_5, fence_3]
})

summary_amount

In [ ]:
#export csv
output_file_amount = OUTPUT_FILES_FOLDER / f"summary_refund_amount_eur_thresholds_per_driver_for_fiscalization_{START_DATE}_{END_DATE}.csv"
summary_amount.to_csv(output_file_amount, index=False)

In [ ]:
thresholds_pct_refunds_from_on_board_sales_and_refunds_per_driver_for_fiscalization = summary_amount.iloc[5, 1]
thresholds_pct_refunds_from_on_board_sales_and_refunds_per_driver_for_fiscalization

In [ ]:
suspected_drivers= aggregated_driver_with_refunds_df[aggregated_driver_with_refunds_df["pct_refunds_from_on_board_sales_and_refunds"] > thresholds_pct_refunds_from_on_board_sales_and_refunds_per_driver_for_fiscalization]

len(suspected_drivers)

In [ ]:
#export csv of suspected drivers

output_file_suspected_drivers = OUTPUT_FILES_FOLDER / f"suspected_drivers_for_fiscalization_{START_DATE}_{END_DATE}.csv"
suspected_drivers.to_csv(output_file_suspected_drivers, index=False)

#export csv of suspected drivers per agency

for agency_id, df_agency in suspected_drivers.groupby("agency_id"):
    output_file = OUTPUT_FILES_FOLDER / f"{agency_id}_suspected_drivers_for_fiscalization_{START_DATE}_{END_DATE}.csv"
    df_agency.to_csv(output_file, index=False)

In [ ]:
# get count of suspected drivers per agency
count_suspected_drivers_by_agency= suspected_drivers.groupby("agency_id").size().reset_index(name='suspected_drivers_count')

## Total refund per Agency

In [ ]:
# Aggregate
aggregated_agency = (
    aggregated_driver_with_refunds_df
    .groupby("agency_id", as_index=False)
    .agg(
        apex_on_board_refunds_qty=("apex_on_board_refunds_qty", "sum"),
        apex_on_board_refunds_amount_eur=("apex_on_board_refunds_amount_eur", "sum"),
        passengers_observed=("passengers_observed", "sum"),
        passengers_observed_on_board_sales_qty=("passengers_observed_on_board_sales_qty", "sum"),
        on_board_sales_and_refunds=("on_board_sales_and_refunds", "sum"),
)
)


# Add count of suspected drivers per agency to aggregated_agency
aggregated_agency = aggregated_agency.merge(count_suspected_drivers_by_agency, on="agency_id", how="left").fillna({"suspected_drivers_count": 0})

# Create CM row as DataFrame
cm_row = pd.DataFrame({
    "agency_id": ["CM"],
    "apex_on_board_refunds_qty": [
        aggregated_agency["apex_on_board_refunds_qty"].sum()
    ],
    "apex_on_board_refunds_amount_eur": [
        aggregated_agency["apex_on_board_refunds_amount_eur"].sum()
    ],
     "passengers_observed": [aggregated_agency["passengers_observed"].sum()],       
      "passengers_observed_on_board_sales_qty": [aggregated_agency["passengers_observed_on_board_sales_qty"].sum()],
    "on_board_sales_and_refunds": [aggregated_agency["on_board_sales_and_refunds"].sum()],
    "suspected_drivers_count": [aggregated_agency["suspected_drivers_count"].sum()]
        
})

# Append
aggregated_agency = pd.concat(
    [aggregated_agency, cm_row],
    ignore_index=True
)

# Now rounding works
aggregated_agency["apex_on_board_refunds_qty"] = aggregated_agency["apex_on_board_refunds_qty"].round(0)
aggregated_agency["apex_on_board_refunds_amount_eur"] = aggregated_agency["apex_on_board_refunds_amount_eur"].round(2)
aggregated_agency["passengers_observed"] = aggregated_agency["passengers_observed"].round(0)
aggregated_agency["passengers_observed_on_board_sales_qty"] = aggregated_agency["passengers_observed_on_board_sales_qty"].round(0)

aggregated_agency["pct_on_board_sales_qty_from_passengers_observed"] = (aggregated_agency["passengers_observed_on_board_sales_qty"]/aggregated_agency["passengers_observed"])
aggregated_agency["pct_on_board_refunds_qty_from_passengers_observed"] = (aggregated_agency["apex_on_board_refunds_qty"]/aggregated_agency["passengers_observed"])

aggregated_agency["pct_refunds_from_on_board_sales_and_refunds"] = (aggregated_agency["apex_on_board_refunds_qty"] / aggregated_agency["on_board_sales_and_refunds"].replace(0, np.nan)).round(4)
#"pct_refunds_from_on_board_sales_and_refunds": [aggregated_agency["pct_refunds_from_on_board_sales_and_refunds"].mean()]
aggregated_agency.head()


In [ ]:
#export csv
output_aggregated_refunds_agency = OUTPUT_FILES_FOLDER / f"aggregated_refunds_by_agency_{START_DATE}_{END_DATE}.csv"
aggregated_agency.to_csv(output_aggregated_refunds_agency, index=False)